# Min-Max Tic-Tac-Toe:

Este trabalho aborda a implementação de dois agentes inteligentes baseados no algoritmo Min-Max, onde há um ambiente multiagente, totalmente observável onde cada agente toma uma decisão baseado em maximizar (ou minimizar) sua utilidade de movimento, enquanto minimiza (ou maximiza) a próxima ação do seu adversário. Um agente tenta maximizar sua utilidade (MAX), enquanto o outro tenta minimizá-la (MIN).

O jogo da velha, por ter um tamanho reduzido, é um ótimo problema que pode ser modelado através desse algoritmo. Utilizei a ideia de árvores de decisão, em que calculamos um score, porém, coloquei funções de fallback, para evitar de escrever mais uma matriz na memória, apenas a fim de calcular o movimento do adversário.

No primeiro agente, além da implementação padrão do Min-Max, foi introduzido um mecanismo inspirado em simulated annealing, utilizando um parâmetro de temperatura para controlar o nível de exploração. Quando a temperatura é maior que zero, o agente não escolhe necessariamente a melhor jogada, mas sim uma jogada baseada em uma distribuição de probabilidade (softmax) sobre os valores retornados pelo Min-Max. Isso permite explorar diferentes caminhos da árvore, evitando comportamento totalmente determinístico de um Climb Hill. Porém, como ele erra muito cedo em um jogo de pequeno tamanho de estados como o Jogo da Velha, tende a perder quando não começa, pois pode desperdiçar através do softmax, os dois primeiros movimentos, e quem começa no jogo da velha tem o benefício de um turno a mais.

Além disso, foi implementado um segundo agente com comportamento subótimo controlado, que ainda utiliza o Min-Max para avaliar os estados, que apenas escolhe com base em Min-Max, o que pode desperdiçar o primeiro movimento quando começa, mas ainda joga bem o suficiente para empatar. Porém, a implementação proposital do agente desperdiçar o primeiro movimento se todos os estados estão vazios, demonstra que mesmo com min max, um erro no primeiro movimento já compromete toda a partida. Como o jogo é determinístico, o oponente ótimo consegue explorar esse erro e garantir vitória ou empate.

## Casos Tratados:
```
    Agente Annealing x Agente First Move Determinístico
    Agente First Move Determinístico x Agente Annealing
    Usuário x Agente Annealing
    Agente Annealing x Usuário
    Usuário x Agente First Move Determinístico
    Agente First Move Determinístico x Usuário
```


In [36]:
import random
import math

class Agent:
    
    def __init__(self, environment):
        self.environment = environment

    def actions(self, state):
        actions = []
        for i in range(3):
            for j in range(3):
                if state[i][j] == 0:
                    actions.append((i, j))
        return actions

    def result(self,state,action,player):
        i, j = action
        state[i][j] = player

    def terminal(self,state):
        return self.utility(state) is not None or not self.actions(state)
    def undo(self, state, action):
        i, j = action
        state[i][j] = 0

    def utility(self, state):
        finish_states = [
            [(0,0),(0,1),(0,2)],
            [(1,0),(1,1),(1,2)],
            [(2,0),(2,1),(2,2)],
            [(0,0),(1,0),(2,0)],
            [(0,1),(1,1),(2,1)],
            [(0,2),(1,2),(2,2)],
            [(0,0),(1,1),(2,2)],
            [(0,2),(1,1),(2,0)]
        ]

        for pos in finish_states:
            a, b, c = pos
            if state[a[0]][a[1]] == state[b[0]][b[1]] == state[c[0]][c[1]] != 0:
                return state[a[0]][a[1]]

        if all(cell != 0 for row in state for cell in row):
            return 0

        return None

class AgentFirstMove(Agent):
    def choose_action(self, state, player):
        if all(cell == 0 for row in state for cell in row):
            return self.actions(state)[0]  # pega o primeiro estado possível

        return move_without_annealing(self, state, player)        

# FirstMoveAgent (seria sempre como temperatura 0 -> determinístico)
def move_without_annealing(agent, state, player):
    actions = agent.actions(state)
    values = []

    for action in actions:
        agent.result(state, action, player)
        value = minimax(agent, state, player == -1, 0)
        agent.undo(state, action)
        values.append(value)
    if player == 1:
        return actions[values.index(max(values))]
    else:        
        return actions[values.index(min(values))]

#Default Agent
def best_move(agent, state, player, temperature):
    actions = agent.actions(state)
    values = []

    for action in actions:
        agent.result(state, action, player)
        value = minimax(agent, state, player == -1, 0)
        agent.undo(state, action)
        values.append(value)

    # temperatura <= 0 = comportamento determinístico
    if temperature <= 0:
        if player == 1:
            return actions[values.index(max(values))]
        else:
            return actions[values.index(min(values))]

    # softmax
    max_v = max(values)
    scaled = [(v - max_v) / temperature for v in values]
    exp_values = [math.exp(v) for v in scaled]
    total = sum(exp_values)
    probs = [v / total for v in exp_values]

    # escolha probabilística
    return random.choices(actions, weights=probs, k=1)[0]

    return best_action

def minimax(agent, state, isMaximizing, depth=0):
    
    if agent.terminal(state):
        result = agent.utility(state)
        if result == 1:
            return 10 - depth
        elif result == -1:
            return depth - 10
        else:
            return 0

    if isMaximizing:
        best = -float("inf")
        for action in agent.actions(state):
            agent.result(state, action, 1)
            value = minimax(agent, state, False, depth + 1)
            agent.undo(state, action)
            best = max(best, value)
        return best
    else:
        best = float("inf")
        for action in agent.actions(state):
            agent.result(state, action, -1)
            value = minimax(agent, state, True, depth + 1)
            agent.undo(state, action)
            best = min(best, value)
        return best

def print_board(state):
    symbols = {1: "X", -1: "O", 0: " "}
    for i in range(3):
        row = [symbols[state[i][j]] for j in range(3)]
        print(" | ".join(row))
        if i < 2:
            print("-" * 5)


def play_game(human_player=1, start_player=1):
    """
    human_player: 1 (X) or -1 (O)
    start_player: who starts (1 or -1)
    """

    state = [[0,0,0],
             [0,0,0],
             [0,0,0]]

    agent = Agent(state)
    current_player = start_player
    temperature = 0.2
    cooling = 0.9

    while True:
        print("\nBOARD:")
        print_board(state)

        if agent.terminal(state):
            result = agent.utility(state)
            if result == 1:
                print("X WINS!")
            elif result == -1:
                print("O WINS!")
            else:
                print("DRAW!")
            break

        if current_player == human_player:
            print("\nYour turn!")
            while True:
                try:
                    i = int(input("Linha (0-2): "))

                    if (str(i).lower() =="quit"):
                        return
                    
                    j = int(input("Coluna (0-2): "))
                    
                    if (str(j).lower() =="quit"):
                        return
                    
                    if (i, j) in agent.actions(state):
                        move = (i, j)
                        break
                    else:
                        print("Invalid position!")
                except:
                    print("Invalid input!")
        else:
            print("\nAGENT PLAYING!")
            
            if all(cell == 0 for row in state for cell in row):
                move = random.choice(agent.actions(state))
            else:
                move = best_move(agent, state, current_player, temperature)
                temperature = temperature - cooling
        agent.result(state, move, current_player)
        current_player *= -1 #alternar o player com base em um estado -1 e 1 

def agent_vs_agent(agent1, agent2, start_player=1):
    state = [[0,0,0],[0,0,0],[0,0,0]]
    current_player = start_player

    base_agent = Agent(state)
    temperature = 0.2
    cooling = 0.1

    while True:
        print("\nBOARD:")
        print_board(state)

        if base_agent.terminal(state):
            result = base_agent.utility(state)
            if result == 1:
                print("X WINS!")
            elif result == -1:
                print("O WINS!")
            else:
                print("DRAW!")
            break

        # escolhe quem joga
        current_agent = agent1 if current_player == 1 else agent2

        # Se é o agente determinístico, usa o método específico, senão usa o método com temperatura
        if hasattr(current_agent, "choose_action"):
            move = current_agent.choose_action(state, current_player)
        else:
            move = best_move(current_agent, state, current_player, temperature)
            temperature =  temperature - cooling
        base_agent.result(state, move, current_player)

        current_player *= -1

def play_game_agent2(human_player=1, start_player=1):
    """
    Joga uma partida entre um humano e o agente AgentFirstMove.
    
    Parâmetros:
    - human_player: 1 (X) ou -1 (O) - qual símbolo o humano controla.
    - start_player: 1 ou -1 - quem começa jogando.
    """
    state = [[0, 0, 0],
             [0, 0, 0],
             [0, 0, 0]]
    
    opponent = AgentFirstMove(state)  
    
    current_player = start_player
    
    while True:
        print("\nBOARD:")
        print_board(state)
        
        if Agent(state).terminal(state):  
            result = Agent(state).utility(state)
            if result == 1:
                print("X WINS!")
            elif result == -1:
                print("O WINS!")
            else:
                print("DRAW!")
            break
        
        if current_player == human_player:
            print("\nYour turn!")
            while True:
                try:
                    i = input("Linha (0-2): ")
                    if i.lower() == "quit":
                        return
                    i = int(i)
                    
                    j = input("Coluna (0-2): ")
                    if j.lower() == "quit":
                        return
                    j = int(j)
                    
                    if (i, j) in Agent(state).actions(state):
                        move = (i, j)
                        break
                    else:
                        print("Invalid position!")
                except ValueError:
                    print("Invalid input! Use numbers between 0 and 2.")
        else:
            print("\nAGENT FIRST MOVE PLAYING!")
            move = opponent.choose_action(state, current_player)
        
        Agent(state).result(state, move, current_player)
        current_player *= -1


In [37]:
if __name__ == "__main__":
    agent_strong = Agent(None)
    agent_weak = AgentFirstMove(None)

    while True:
        print("\nMENU:")
        print("0 - QUIT")
        print("1 - AGENT1 (ANNEALING) vs AGENT2 (FirstMove)")
        print("2 - AGENT2 (FirstMove) vs AGENT1 (ANNEALING) ")
        print("3 - AGENT1 (ANNEALING)  vs USER")
        print("4 - USER vs AGENT1 (ANNEALING) ")
        print("5 - AGENT2 (FirstMove) vs USER")
        print("6 - USER vs AGENT2 (FirstMove)")

        options = input("Escolha: ")

        if options == "0":
            break

        try:
            if options == "1":
                agent_vs_agent(agent_strong, agent_weak, start_player=1)
            elif options == "2":
                agent_vs_agent(agent_weak, agent_strong, start_player=1)
            elif options == "3":
                play_game(human_player=-1, start_player=1)  
            elif options == "4":
                play_game(human_player=1, start_player=1)
            elif options == "5":
                play_game_agent2(human_player=-1, start_player=1)  
            elif options == "6":
                play_game_agent2(human_player=1, start_player=1)   

        except KeyboardInterrupt:
            print("\GAME INTERRUPTED.")

<>:35: SyntaxWarning: invalid escape sequence '\G'
<>:35: SyntaxWarning: invalid escape sequence '\G'
C:\Users\User\AppData\Local\Temp\ipykernel_31204\2294035918.py:35: SyntaxWarning: invalid escape sequence '\G'
  print("\GAME INTERRUPTED.")



MENU:
0 - QUIT
1 - AGENT1 (ANNEALING) vs AGENT2 (FirstMove)
2 - AGENT2 (FirstMove) vs AGENT1 (ANNEALING) 
3 - AGENT1 (ANNEALING)  vs USER
4 - USER vs AGENT1 (ANNEALING) 
5 - AGENT2 (FirstMove) vs USER
6 - USER vs AGENT2 (FirstMove)

BOARD:
  |   |  
-----
  |   |  
-----
  |   |  

Your turn!

BOARD:
X |   |  
-----
  |   |  
-----
  |   |  

AGENT PLAYING!

BOARD:
X |   |  
-----
  |   |  
-----
  |   | O

Your turn!

BOARD:
X | X |  
-----
  |   |  
-----
  |   | O

AGENT PLAYING!

BOARD:
X | X | O
-----
  |   |  
-----
  |   | O

Your turn!

BOARD:
X | X | O
-----
  |   | X
-----
  |   | O

AGENT PLAYING!

BOARD:
X | X | O
-----
  |   | X
-----
O |   | O

Your turn!

BOARD:
X | X | O
-----
  | X | X
-----
O |   | O

AGENT PLAYING!

BOARD:
X | X | O
-----
  | X | X
-----
O | O | O
O WINS!

MENU:
0 - QUIT
1 - AGENT1 (ANNEALING) vs AGENT2 (FirstMove)
2 - AGENT2 (FirstMove) vs AGENT1 (ANNEALING) 
3 - AGENT1 (ANNEALING)  vs USER
4 - USER vs AGENT1 (ANNEALING) 
5 - AGENT2 (FirstMove) vs 